# 🧠 DietBot: PDF-Grounded Question Generation and Evaluation

This notebook generates realistic, emotionally authentic questions grounded in authoritative PDF content (e.g., ADA guidelines), and verifies whether "answerable" questions are actually supported by the reference documents using a retrieval-augmented LLM.

---

## 📌 What It Does

1. **Ingests** all PDF files from `data/input_pdfs/`
2. **Chunks & Embeds** documents using LangChain + FAISS
3. **Generates** 100 natural-language questions (50 answerable, 50 unanswerable) using GPT-4o
4. **Verifies** answerable questions using GPT-4o against the original documents
5. **Saves** results to `data/csv_outputs/`

## 🚀 How to Use

1. **Set your OpenAI API key**

Create a `.env` file in the `notebooks/` directory with:

```env
OPENAI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
```

Run cells in order

In [ ]:
!pip install pandas
!pip install --upgrade langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.68
    Uninstalling langchain-core-0.3.68:
      Successfully uninstalled langchain-core-0.3.68
  Attempting uninstall: langchain-text-splitters 0/3 [langchain-core]
    Found existing installation: langchain-text-splitters 0.3.8langchain-core]
    Uninstalling langchain-text-splitters-0.3.8: 0/3 [langchain-core]
      Successfully uninstalled langchain-text-splitters-0.3.8 [langchain-core]
  Attempting uninstall: langchain━━━━━━━━━━━ 0/3 [langchain-core]
    Found existing installation: langchain 0.3.260/3 [langchain-core]
    Uninstalling langchain-0.3.26:╸━━━━━━━━━━━━━ 2/3 [langchain]
      Successfully uninstalled langchain-0.3.260m━━━━━━━━━━━━━ 2/3 [langchain]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain]/3 [langchain]


### 🧠 Run a Local LLM with Ollama (Gemma 3)

This cell sends a prompt to a **locally running Ollama instance**, using the `requests` library to call the `/api/generate` endpoint.

**What it does:**
- Connects to `http://localhost:11434`
- Sends a sample health-related prompt to the `gemma3:4b-it-qat` model:
  > _"Can you manage diabetes just with food and no meds?"_
- If the call succeeds, you'll see the model's response printed below.

#### ⚙️ Before you run this:
- Make sure you have [Ollama](https://ollama.com) installed and running locally
- You should have the model pulled (e.g., `gemma3:4b-it-qat`)  
  Pull it by running:
  ```bash
  ollama pull gemma3:4b-it-qat
  ```
- If you're using a remote or hosted Ollama server, **update the `OLLAMA_API_URL`** to match your endpoint

✅ If everything works, you’ll see a generated answer to the question printed in the cell output.


In [8]:
import requests

OLLAMA_API_URL = "http://localhost:11434/api/generate"

payload = {
    "model": "gemma3:4b-it-qat",  # or whatever model you pulled
    "prompt": "Can you manage diabetes just with food and no meds?",
    "stream": False
}

response = requests.post(OLLAMA_API_URL, json=payload)

if response.ok:
    result = response.json()
    print(result["response"])
else:
    print(f"Error: {response.status_code} - {response.text}")

Okay, let's tackle this important and complex question. The short answer is: **it *can* be managed, but it's incredibly challenging and not suitable for everyone.** It's definitely possible to manage type 2 diabetes with diet and lifestyle changes alone, but it requires a huge commitment, careful planning, and ongoing monitoring. Here's a breakdown of what’s involved and why it's not a guaranteed solution:

**1. Type 2 Diabetes and the Role of Food & Lifestyle:**

* **Insulin Resistance:** Type 2 diabetes is largely driven by insulin resistance – where your body's cells don't respond effectively to insulin. Insulin is the hormone that helps glucose (sugar) from your blood enter your cells to be used for energy.
* **Diet's Impact:** Diet plays a *massive* role in controlling insulin resistance.  
    * **Carbohydrates:** Managing carb intake is crucial. Focusing on complex carbs (whole grains, legumes, vegetables) over simple carbs (sugary drinks, white bread) helps stabilize blood suga

## 📋 Manually Constructing a Baseline Question Dataset for Evaluation

This cell creates a clean and structured DataFrame of **100 human-like questions** designed for evaluating a diabetes-focused chatbot system. These questions are divided into two balanced categories:

### ✅ 1. Answerable Questions (50)
- These reflect realistic, practical concerns that *can* be grounded in evidence from authoritative sources (e.g., ADA PDFs).
- Topics include food choices, lifestyle management, and common symptoms.
- Example: *“Can I still eat dessert if I plan my meals right?”*

### ❌ 2. Unanswerable / Speculative Questions (50)
- These represent misinformation, magical thinking, or extreme edge cases.
- They are designed to test whether the chatbot resists hallucination or speculation.
- Example: *“Can grounding (walking barefoot) balance my insulin levels?”*

### 🔨 How the Code Works

1. **Two lists** are defined: `answerable_questions` and `unanswerable_questions`, each with 50 items.
2. A combined list `all_questions` is created, along with `question_types` to label them.
3. Unique IDs (`Q001` to `Q100`) are generated using `zfill(3)` formatting.
4. A `pandas.DataFrame` is assembled with three columns:
   - `Question ID`
   - `Question Type`
   - `Question`
5. The first 100 rows are printed to verify structure and format.

This forms a **ground truth dataset** that can now be:
- Passed through an LLM Judge for FactCheck and Empathy scoring
- Used for training or fine-tuning chatbot models
- Exported to CSV for further experiments

> ✍️ Note: This is manually curated and reflects **natural-sounding, emotionally grounded questions** — not synthetic or academic language.


In [13]:
import pandas as pd

# 50 answerable questions
answerable_questions = [
    "Can you manage diabetes just with food and no meds?",
    "Is walking after meals actually helpful for blood sugar?",
    "Do probiotics help with blood sugar control?",
    "Should I avoid bananas or are they okay in moderation?",
    "Can I still eat carbs if I'm trying to lower my A1C?",
    "How many grams of sugar should I aim for in a day if I’m prediabetic?",
    "What’s the best breakfast for someone with type 2 diabetes?",
    "Are low-carb diets the only way to manage diabetes?",
    "What is the best way to manage cravings?",
    "Can I still eat dessert if I plan my meals right?",
    "Can drinking alcohol mess with my blood sugar readings?",
    "Is oatmeal okay or does it spike blood sugar too much?",
    "Are cheat days okay if I eat healthy most of the week?",
    "Should I eat the same number of carbs at every meal?",
    "Can weight loss actually reverse prediabetes?",
    "What should I do if I feel dizzy or shaky between meals?",
    "How can I prevent diabetes complications like nerve damage?",
    "Do I need to count net carbs or total carbs?",
    "Is eating out possible, or should I always cook at home?",
    "How long does it take to see A1C improvement after changing your diet?",
    "Can I eat pasta if I pair it with vegetables and protein?",
    "Is coffee bad for blood sugar control?",
    "Do protein shakes spike blood sugar?",
    "How does fiber help with blood sugar?",
    "How can I talk to my family about changing our eating habits?",
    "Do I need to take supplements like magnesium or cinnamon?",
    "What’s a safe blood sugar range after eating?",
    "Are sugar substitutes safe to use daily?",
    "Are CGMs worth it for someone who’s not on insulin?",
    "Do I have to give up rice completely?",
    "How much water should I drink each day to help manage blood sugar?",
    "Can skipping meals make my blood sugar worse?",
    "Can high cholesterol make diabetes worse?",
    "Are there smartphone apps that actually help with managing diabetes?",
    "Do I need to see a nutritionist or can I figure this out on my own?",
    "If my fasting glucose is okay but my A1C is high, what does that mean?",
    "What’s the difference between type 1 and type 2 diabetes?",
    "Are there cultural foods I can still eat while managing diabetes?",
    "What is the role of weight loss in achieving diabetes remission?",
    "What’s the best type of exercise for people with diabetes?",
    "Can I eat fruit if I watch my portions?",
    "Is brown rice better than white rice for blood sugar?",
    "How do I handle cravings for sweets in the evening?",
    "Can I eat bread if it's whole grain?",
    "How does stress affect my blood sugar levels?",
    "What kind of doctor should I see for prediabetes?",
    "Are there any easy meal prep ideas for managing diabetes?",
    "How much exercise per week helps with insulin resistance?",
    "Is it okay to drink diet soda if I'm diabetic?",
    "What is insulin resistance and how do I know if I have it?"
]

# 50 unanswerable questions
unanswerable_questions = [
    "Can I use apple cider vinegar to never get diabetes again?",
    "If I manifest better health, will my A1C improve?",
    "Can grounding (walking barefoot) balance my insulin levels?",
    "Will prayer alone replace insulin?",
    "Can eating only raw foods reverse diabetes permanently?",
    "If I cut out all sugar forever, will I become immune to diabetes?",
    "Can dreams predict blood sugar spikes?",
    "If I move to a tropical island, will my diabetes disappear?",
    "Is there one superfood that prevents diabetes?",
    "Can positive affirmations change my glucose curve?",
    "Will talking to plants reduce my cravings?",
    "Is diabetes punishment for past bad eating?",
    "Can I detox my pancreas by drinking lemon water for a week?",
    "If I fast for 3 days straight, will my diabetes reset?",
    "Will being vegan make me invincible to blood sugar issues?",
    "Is diabetes a government conspiracy?",
    "If I lose enough weight, can I delete my diabetes genes?",
    "Will getting rid of my microwave lower my A1C?",
    "Can I pass prediabetes to my kids through my bloodline?",
    "Is eating at the exact same time every day a diabetes cure?",
    "Can astrology tell me if I’ll get diabetes?",
    "Can sound therapy lower my blood sugar?",
    "Is wearing blue clothes good for my pancreas?",
    "If I walk backwards, will that improve insulin function?",
    "Can I train my body to not respond to carbs?",
    "Is prediabetes just made up by pharma to sell drugs?",
    "Will the new diabetes vaccine cure me instantly?",
    "Can aliens help us fix diabetes with their tech?",
    "Will a cold shower fix insulin resistance?",
    "Can I heal my beta cells by thinking positively?",
    "Is sugar evil and should I fear it like poison?",
    "Can I cure diabetes with essential oils?",
    "If I sleep 10 hours a night, will I reverse diabetes?",
    "Will breathing exercises change my glucose levels permanently?",
    "Is there a tea that makes insulin unnecessary?",
    "Will moving to the mountains make me diabetes-free?",
    "If my mom had gestational diabetes, does that mean I’m doomed?",
    "Can eating only purple food reverse prediabetes?",
    "Can I vibe my way to healthy blood sugar?",
    "Can dancing daily replace metformin?",
    "Can sunlight cure diabetes?",
    "Does singing before meals reduce insulin spikes?",
    "Can I permanently fix my pancreas with smoothies?",
    "Is there a moon phase that lowers A1C?",
    "Will avoiding shoes improve my glucose levels?",
    "Can standing on one leg daily reverse insulin resistance?",
    "If I only eat cold foods, will I avoid diabetes?",
    "Is diabetes caused by bad karma?",
    "Can my pet's diet affect my diabetes?",
    "Will thinking about sugar cause a spike in my blood glucose?"
]

# Combine questions and types
all_questions = answerable_questions + unanswerable_questions
question_types = ["Answerable"] * len(answerable_questions) + ["Unanswerable"] * len(unanswerable_questions)

# Generate question IDs: Q001 to Q100
question_ids = [f"Q{str(i+1).zfill(3)}" for i in range(len(all_questions))]

# Build DataFrame with Question ID
df = pd.DataFrame({
    "Question ID": question_ids,
    "Question Type": question_types,
    "Question": all_questions
})

# Display first 100 rows
print(df.head(100))

   Question ID Question Type  \
0         Q001    Answerable   
1         Q002    Answerable   
2         Q003    Answerable   
3         Q004    Answerable   
4         Q005    Answerable   
..         ...           ...   
95        Q096  Unanswerable   
96        Q097  Unanswerable   
97        Q098  Unanswerable   
98        Q099  Unanswerable   
99        Q100  Unanswerable   

                                             Question  
0   Can you manage diabetes just with food and no ...  
1   Is walking after meals actually helpful for bl...  
2        Do probiotics help with blood sugar control?  
3   Should I avoid bananas or are they okay in mod...  
4   Can I still eat carbs if I'm trying to lower m...  
..                                                ...  
95  Can standing on one leg daily reverse insulin ...  
96   If I only eat cold foods, will I avoid diabetes?  
97                   Is diabetes caused by bad karma?  
98              Can my pet's diet affect my diabetes?  

In [1]:
import pandas as pd
from langchain.schema import Document

# Load selected columns from CSV
df = pd.read_csv("answerable_qna.csv", usecols=["Question ID", "Question Type", "Question"])

# Convert to LangChain Document objects
documents = [
    Document(
        page_content=row["Question"],
        metadata={
            "Question ID": row["Question ID"],
            "Question Type": row["Question Type"]
        }
    )
    for _, row in df.iterrows()
]

# Confirm load
print(f"✅ Loaded {len(documents)} questions.")
print("🔍 Sample:", documents[0].page_content)

✅ Loaded 50 questions.
🔍 Sample: What's the best diet to lower my A1C?


## 🤖 Unified Answer Generator for Multiple LLMs

This cell defines a function called `get_answer(question, model_name)` that sends a diabetes-related question to one of several supported **chat models** and returns a plain-language, empathetic response.

### 🧠 Function: `get_answer(question, model_name="gpt-4")`

#### 🔶 Inputs:
- `question` – A natural-language user question (e.g., *"Can I eat fruit if I'm prediabetic?"*)
- `model_name` – The model to use. Supported values:
  - `"gpt-4"` – OpenAI GPT-4
  - `"gpt-3.5"` – OpenAI GPT-3.5 Turbo
  - `"claude-3"` – Anthropic Claude 3 Opus
  - `"gemini-pro"` – Google Gemini 1 Pro (via PaLM API)
  - `"gemma3"` – Local Ollama model (e.g., `gemma:3b`)

#### 🧾 Prompt Structure:
- A **SystemMessage** establishes a consistent assistant persona:
  > "You are a friendly, empathetic assistant helping people with diabetes or prediabetes..."
- A **HumanMessage** presents the question in a Q&A format:
  > `Q: {question} \n A:`

This helps standardize tone and formatting across different models.

### 🔌 Model Routing:
The function initializes the correct LLM wrapper from LangChain based on the `model_name`:
- 🧠 GPT models via `ChatOpenAI`
- 🦉 Claude via `ChatAnthropic`
- 🌟 Gemini via `ChatGooglePalm`
- 🧪 Ollama-hosted local models via `ChatOllama`

If the model name is unsupported, it raises an error.

### ✅ Output:
Returns a clean, stripped string of the LLM’s response — ready to be evaluated, displayed, or saved.

> ⚠️ **Note:** You must have local Ollama running for `"gemma3"` to work. Adjust the model name (`gemma:3b`) to match your local pull if needed.


In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain_community.chat_models import ChatAnthropic, ChatGooglePalm
from langchain_community.chat_models import ChatOllama
from langchain.schema import HumanMessage, SystemMessage

def get_answer(question, model_name="gpt-4"):
    system_prompt = (
        "You are a friendly, empathetic assistant helping people with diabetes "
        "or prediabetes. Provide clear, supportive, evidence-based answers in plain language."
    )

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Q: {question}\nA:")
    ]

    if model_name == "gpt-4":
        llm = ChatOpenAI(model="gpt-4", temperature=0)
    elif model_name == "gpt-3.5":
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    elif model_name == "claude-3":
        llm = ChatAnthropic(model="claude-3-opus-20240229", temperature=0)
    elif model_name == "gemini-pro":
        llm = ChatGooglePalm(model="models/chat-bison-001", temperature=0)
    elif model_name == "gemma3":
        llm = ChatOllama(model="gemma:3b", temperature=0)  # adjust to your pulled model name
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return llm(messages).content.strip()

## 🧠 Unified Answer Generator (LangChain + Ollama Direct)

This cell defines a versatile function `get_answer(question, model_name)` that supports **both cloud-based LLMs** (via LangChain) and **local models** hosted on Ollama through direct HTTP requests.

---

### 🔄 Purpose
To query various LLMs with a consistent, friendly system prompt designed for a diabetes-focused health assistant.

---

### 💬 System Prompt
All models receive the following persona instructions:

> "You are a friendly, empathetic assistant helping people with diabetes or prediabetes. Provide clear, supportive, evidence-based answers in plain language."

---

### 🧩 Model Routing Logic

#### 🔷 `model_name = "gemma3"` → Direct Ollama HTTP API
- Uses `requests.post()` to send a prompt to a local Ollama server (e.g. `zitro-box-1:11434`)
- Custom JSON payload:
  - `model`: Must match the local Ollama model name (e.g., `"gemma3:4b-it-qat"`)
  - `prompt`: Combines system prompt and user question
  - `stream`: Set to `False` for full response in one go
- Handles errors if the local server is down or misconfigured

📍 **Important:**  
Ollama must be running and reachable at the specified `OLLAMA_API_URL`.

---

#### 🔷 All Other Models → LangChain Wrappers
- Converts system/user prompt into `SystemMessage` and `HumanMessage`
- Routes to one of the following:
  - `gpt-4` → OpenAI GPT-4 via LangChain
  - `gpt-3.5` → OpenAI GPT-3.5 Turbo
  - `claude-3` → Anthropic Claude 3 Opus
  - `gemini-pro` → Google Gemini 1 Pro (chat-bison)
- Raises a `ValueError` for unsupported model names

---

### ✅ Output
Returns the model's answer as a clean text string (no metadata or JSON).

---

### 🔧 Example Call
```python
get_answer("Can I still eat rice if I'm prediabetic?", model_name="gemma3")


In [3]:
import requests
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain_community.chat_models import ChatAnthropic, ChatGooglePalm
from langchain.schema import HumanMessage, SystemMessage

OLLAMA_API_URL = "http://localhost:11434/api/generate"

def get_answer(question, model_name="gpt-4"):
    system_prompt = (
        "You are a friendly, empathetic assistant helping people with diabetes "
        "or prediabetes. Provide clear, supportive, evidence-based answers in plain language."
    )

    if model_name == "gemma3":
        # Use direct Ollama HTTP request
        prompt = f"{system_prompt}\n\nQ: {question}\nA:"
        payload = {
            "model": "gemma3:4b-it-qat",  # adjust if using a different model name
            "prompt": prompt,
            "stream": False
        }

        response = requests.post(OLLAMA_API_URL, json=payload)
        if response.ok:
            return response.json()["response"].strip()
        else:
            raise RuntimeError(f"Ollama error: {response.status_code} - {response.text}")

    else:
        # Use LangChain LLMs
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Q: {question}\nA:")
        ]

        if model_name == "gpt-4":
            llm = ChatOpenAI(model="gpt-4", temperature=0)
        elif model_name == "gpt-3.5":
            llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        elif model_name == "claude-3":
            llm = ChatAnthropic(model="claude-3-opus-20240229", temperature=0)
        elif model_name == "gemini-pro":
            llm = ChatGooglePalm(model="models/chat-bison-001", temperature=0)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

        return llm(messages).content.strip()


In [4]:
#test get_answer()
print(get_answer("Can I still eat dessert if I plan my meals right?", model_name="gemma3"))


Okay, let's talk about dessert and diabetes/prediabetes! You're asking a really important and common question, and the good news is – **yes, you absolutely can still enjoy dessert sometimes, even with diabetes or prediabetes!** It’s not about completely eliminating it, but it’s about doing it in a smart and mindful way.

Here’s a breakdown of how to approach dessert safely and deliciously:

**1. Portion Control is Key:**

*   **Small Amounts:** Think bite-sized portions. A small scoop of ice cream, a few squares of dark chocolate, or a mini cookie can be enjoyed without derailing your overall meal plan.
*   **Measure it out:** Using measuring spoons and cups can help you stick to those smaller portions.

**2. Choose Wisely – Focus on Lower-Glycemic Options:**

*   **Dark Chocolate (70% cacao or higher):** Dark chocolate has a lower glycemic index (GI) than milk chocolate, meaning it doesn’t raise your blood sugar as quickly.
*   **Fruit-Based Desserts:** Baked apples with cinnamon, ber

In [25]:
import json
import requests
import time
from google.oauth2 import service_account
from google.auth.transport.requests import Request as GoogleAuthRequest
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain_community.chat_models import ChatAnthropic, ChatGooglePalm
from langchain.schema import HumanMessage, SystemMessage

# === Cloud Run Configuration ===
CLOUD_RUN_URL = "https://gemma3-1b-j7lsjhfgua-uc.a.run.app"
ENDPOINT = f"{CLOUD_RUN_URL}/api/generate"
SERVICE_ACCOUNT_FILE = "dietbot-gemma-sa.json"
AUDIENCE = CLOUD_RUN_URL  # Needed for IDTokenCredentials

def get_answer2(question, model_name="gpt-4"):
    system_prompt = (
        "You are a friendly, empathetic assistant helping people with diabetes "
        "or prediabetes. Provide clear, supportive, evidence-based answers in plain language."
        "Lastly, make the response as concise as possible.  Remember 'TLDR;"
    )

    if model_name == "gemma3":
        # Construct prompt and payload
        prompt = f"{system_prompt}\n\nQ: {question}\nA:"
        payload = {
            "model": "gemma3:1b",
            "prompt": prompt,
            "stream": False  # Set to True if you want to stream like original example
        }

        # Get ID token for Cloud Run auth
        credentials = service_account.IDTokenCredentials.from_service_account_file(
            SERVICE_ACCOUNT_FILE, target_audience=AUDIENCE
        )
        credentials.refresh(GoogleAuthRequest())
        token = credentials.token

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        # POST request to Cloud Run endpoint
        response = requests.post(ENDPOINT, headers=headers, json=payload)
        if response.ok:
            return response.json().get("response", "").strip()
        else:
            raise RuntimeError(f"Cloud Run error: {response.status_code} - {response.text}")

    else:
        # Use LangChain-compatible models
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Q: {question}\nA:")
        ]

        if model_name == "gpt-4":
            llm = ChatOpenAI(model="gpt-4", temperature=0)
        elif model_name == "gpt-3.5":
            llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        elif model_name == "claude-3":
            llm = ChatAnthropic(model="claude-3-opus-20240229", temperature=0)
        elif model_name == "gemini-pro":
            llm = ChatGooglePalm(model="models/chat-bison-001", temperature=0)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

        return llm(messages).content.strip()


In [28]:
print(get_answer2("Can I still eat dessert if I plan my meals right?", model_name="gemma3"))


A: Absolutely! Planning your meals with diabetes or prediabetes in mind can help manage your blood sugar. It’s about balancing your food intake with your medications and activity levels. 

**TLDR;** Yes, plan your meals, but don’t restrict yourself completely. Focus on healthy choices!


## 🧪 Generate and Save Answers for 100 Diabetes Questions Using an LLM

This cell runs through 100 pre-written questions (50 answerable and 50 unanswerable) and generates chatbot-style answers using a specified model, such as `gemma3`. It logs progress, tracks timing, and saves the results to a CSV.

---

### 💡 What This Code Does

1. **Loads and Iterates Through Each Question**  
   For each row in the DataFrame `df`, it extracts:
   - `Question ID` (e.g., Q001)
   - `Question Type` ("Answerable" or "Unanswerable")
   - The actual `Question` text

2. **Generates Answers**  
   Calls the `get_answer()` function using the model `"gemma3"` to generate a plain-language, empathetic response. If the model fails, an error is logged instead.

3. **Logs Progress and Time Estimates**  
   For each question:
   - Shows its position in the queue (e.g., `[4/100]`)
   - Shows how long processing has taken so far
   - Estimates remaining time and total duration

4. **Stores the Output**  
   Saves the result as a list of dictionaries with:
   - `Question ID`
   - `Question Type`
   - `Question`
   - `Answer`

   It then converts this list into a DataFrame and writes it to `diabetes_qna_gemma3.csv`.

---

### 📝 Notes
- The model used can be changed by updating the `model_name` in `get_answer()`. Supported options include `"gpt-4"`, `"gemma3"`, `"claude-3"`, and more.
- This script assumes `df` has already been created with all 100 questions and that the `get_answer()` function is defined.

---

\\


In [31]:
import pandas as pd
import time
from datetime import timedelta

# Track results and timing
results = []
total = len(df)

start_time = time.time()

print(f"🚀 Starting processing of {total} questions...\n")

for i, row in df.iterrows():
    question = row["Question"]
    qtype = row["Question Type"]
    qid = row["Question ID"]

    current_index = i + 1
    print(f"[{current_index}/{total}] ⏳ Processing {qid} ({qtype}): \"{question[:60]}...\"")

    try:
        answer = get_answer2(question, model_name="gemma3")
    except Exception as e:
        answer = f"Error: {e}"

    results.append({
        "Question ID": qid,
        "Question Type": qtype,
        "Question": question,
        "Answer": answer
    })

    # Timing calculations
    elapsed = time.time() - start_time
    avg_time = elapsed / current_index
    remaining = total - current_index
    eta = remaining * avg_time
    total_est = avg_time * total

    print(
        f"✅ {qid} done.\n"
        f"   Elapsed: {timedelta(seconds=int(elapsed))} | "
        f"Remaining: {timedelta(seconds=int(eta))} | "
        f"Est. Total: {timedelta(seconds=int(total_est))}\n"
    )

# Save results to CSV
output_df = pd.DataFrame(results)
output_df.to_csv("data/csv_outputs/diabetes_qna_gemma3-wTLDR.csv", index=False)

total_time = time.time() - start_time
print(f"🎉 All done in {timedelta(seconds=int(total_time))}! Results saved to diabetes_qna_gemma3.csv.")


🚀 Starting processing of 50 questions...

[1/50] ⏳ Processing 1 (Answerable): "What's the best diet to lower my A1C?..."
✅ 1 done.
   Elapsed: 0:00:01 | Remaining: 0:00:49 | Est. Total: 0:00:50

[2/50] ⏳ Processing 2 (Answerable): "Can losing weight help reverse my type 2 diabetes?..."
✅ 2 done.
   Elapsed: 0:00:02 | Remaining: 0:00:50 | Est. Total: 0:00:52

[3/50] ⏳ Processing 3 (Answerable): "Is it okay to drink alcohol if I have diabetes?..."
✅ 3 done.
   Elapsed: 0:00:03 | Remaining: 0:00:47 | Est. Total: 0:00:50

[4/50] ⏳ Processing 4 (Answerable): "Will intermittent fasting help my blood sugar?..."
✅ 4 done.
   Elapsed: 0:00:04 | Remaining: 0:00:46 | Est. Total: 0:00:50

[5/50] ⏳ Processing 5 (Answerable): "How many carbs should I eat with type 2 diabetes?..."
✅ 5 done.
   Elapsed: 0:00:04 | Remaining: 0:00:44 | Est. Total: 0:00:49

[6/50] ⏳ Processing 6 (Answerable): "Are low-carb diets safe for diabetics?..."
✅ 6 done.
   Elapsed: 0:00:06 | Remaining: 0:00:45 | Est. Total: 0:00

## 💬 Generate Diabetes Q&A Pairs Using Gemma3 and Save to CSV

This code runs all questions through the `gemma3` model and stores the model's answers in a CSV file. It's a lightweight version of the longer evaluation loop — ideal for quick generation without time tracking or progress reporting.

---

### 🔄 What It Does

1. **Iterates Over Each Row in the DataFrame `df`**  
   For each question:
   - Retrieves the `Question ID`, `Question Type`, and the actual `Question` text

2. **Generates an Answer**  
   Calls the `get_answer()` function with `model_name="gemma3"` to generate a chatbot-style response.
   - If the model call fails, it logs the error message in the `Answer` field.

3. **Collects Results**  
   Builds a list of dictionaries, each containing:
   - `Question ID`
   - `Question Type`
   - `Question`
   - `Answer` (either a valid response or error message)

4. **Saves to CSV**  
   Converts the list to a new DataFrame called `output_df`  
   Then writes it to a file: `diabetes_qna_gemma3.csv`

---

### ✅ Output File Structure

| Question ID | Question Type | Question                                     | Answer                         |
|-------------|----------------|----------------------------------------------|--------------------------------|
| Q001        | Answerable     | Can you manage diabetes just with food...?  | Yes, nutrition therapy...      |
| Q002        | Unanswerable   | Can astrology predict blood sugar spikes?   | There is no scientific basis...|

This script provides a full dataset of question-answer pairs that can be used for chatbot prototyping, evaluation, or training.


In [30]:
import pandas as pd

# Run questions through Gemma3 and store in list
results = []

for i, row in df.iterrows():
    question = row["Question"]
    qtype = row["Question Type"]
    qid = row["Question ID"]
    
    try:
        answer = get_answer(question, model_name="gemma3")
    except Exception as e:
        answer = f"Error: {e}"
    
    results.append({"Question ID": qid, "Question Type": qtype, "Question": question, "Answer": answer})

# Convert to DataFrame and save as CSV
output_df = pd.DataFrame(results)
output_df.to_csv("data/csv_outputs/diabetes_qna_gemma3-wTLDR.csv", index=False)
